In [0]:
from pyspark.sql import functions as F

# Lectura de tabla inicial
df = spark.table("citibike.02_silver.citibikes")

################## Dim time ##################

# Para días en español
days = F.create_map(
    F.lit(1), F.lit("domingo"),
    F.lit(2), F.lit("lunes"),
    F.lit(3), F.lit("martes"),
    F.lit(4), F.lit("miércoles"),
    F.lit(5), F.lit("jueves"),
    F.lit(6), F.lit("viernes"),
    F.lit(7), F.lit("sábado")
)


# Para meses en español
months = F.create_map(
    *[
        x
        for i, m in enumerate(
            ["enero", "febrero", "marzo", "abril", "mayo", "junio",
             "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"],
            1
        )
        for x in (F.lit(i), F.lit(m))
    ]
)

dim_time = (
    df.select(
        F.col("started_at").alias("start_datetime"),
        F.to_date("started_at").alias("start_date"),
        F.year("started_at").alias("year"),
        F.month("started_at").alias("month"),
        F.dayofmonth("started_at").alias("day"),
        F.hour("started_at").alias("hour"),
        F.minute("started_at").alias("minute"),
        F.dayofweek("started_at").alias("day_of_week"),
        days[F.dayofweek("started_at")].alias("day_name"),
        months[F.month("started_at")].alias("month_name")
    )
    .distinct()
    .withColumn(
        "time_id",
        F.date_format("start_datetime", "yyyyMMddHHmmss")
    )
)

################## Dim station ##################
start_stations = df.select(
    F.col("start_station_id").alias("station_id"),
    F.col("start_station_name").alias("station_name"),
    F.col("start_lat").alias("latitude"),
    F.col("start_lng").alias("longitude")
)

end_stations = df.select(
    F.col("end_station_id").alias("station_id"),
    F.col("end_station_name").alias("station_name"),
    F.col("end_lat").alias("latitude"),
    F.col("end_lng").alias("longitude")
)

dim_station = (
    start_stations
    .union(end_stations)
    .filter(F.col("station_id").isNotNull())
    .dropDuplicates(["station_id"])
)


dim_station = dim_station.withColumn(
    "station_key",
    F.monotonically_increasing_id()
)


################## Dim bike type ##################
dim_bike_type = (
    df.select("rideable_type")
    .distinct()
    .filter(F.col("rideable_type").isNotNull())
    .withColumn(
        "bike_type_key",
        F.monotonically_increasing_id()
    )
)


################## Dim member type ##################
dim_member_type = (
    df.select("member_casual")
    .distinct()
    .filter(F.col("member_casual").isNotNull())
    .withColumn(
        "member_type_key",
        F.monotonically_increasing_id()
    )
)

################## Fact trips ##################
df_trips = df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("ended_at") -
        F.unix_timestamp("started_at")
    ) / 60
)

fact_trips = (
    df_trips.alias("f")
    .join(
        dim_bike_type.alias("b"),
        F.col("f.rideable_type") == F.col("b.rideable_type"),
        "left"
    )
    .join(
        dim_member_type.alias("m"),
        F.col("f.member_casual") == F.col("m.member_casual"),
        "left"
    )
    .join(
        dim_station.alias("s"),
        F.col("f.start_station_id") == F.col("s.station_id"),
        "left"
    )
    .join(
        dim_station.alias("e"),
        F.col("f.end_station_id") == F.col("e.station_id"),
        "left"
    )
    .select(
        F.col("f.ride_id"),
        F.col("f.started_at"),
        F.col("f.ended_at"),
        F.col("b.bike_type_key"),
        F.col("m.member_type_key"),
        F.col("s.station_key").alias("start_station_key"),
        F.col("e.station_key").alias("end_station_key"),
        F.col("f.trip_duration_minutes")
    )
)
print('Sample resultado final tabla de hechos:')
display(fact_trips.limit(10))

In [0]:
################## Escrituras de tablas ##################
dim_time.write.mode("overwrite").saveAsTable(
    "citibike.03_gold.dim_time"
)

dim_station.write.mode("overwrite").saveAsTable(
    "citibike.03_gold.dim_station"
)

dim_bike_type.write.mode("overwrite").saveAsTable(
    "citibike.03_gold.dim_bike_type"
)

dim_member_type.write.mode("overwrite").saveAsTable(
    "citibike.03_gold.dim_member_type"
)

fact_trips.write.mode("overwrite").saveAsTable(
    "citibike.03_gold.fact_trips"
)